## Iniciando o Spark e importando bibliotecas

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("AnaliseDadosPrevious") \
    .master("spark://spark-master:7077") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/13 16:06:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
previous_application = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/data/raw/previous_application.csv")
previous_application.createOrReplaceTempView("previous_application")

previous_application.show(5, truncate=False)

25/07/13 16:06:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+----------+------------------+-----------+---------------+----------+----------------+---------------+--------------------------+-----------------------+---------------------------+----------------------+-----------------+---------------------+------------------------+----------------------+--------------------+-------------+---------------------+------------------+---------------+----------------+-------------------+--------------+-----------------+-----------------------+----------------+--------------------+-----------+----------------+------------------------+------------------+--------------+-------------------------+-------------+----------------+-------------------------+
|SK_ID_PREV|SK_ID_CURR|NAME_CONTRACT_TYPE|AMT_ANNUITY|AMT_APPLICATION|AMT_CREDIT|AMT_DOWN_PAYMENT|AMT_GOODS_PRICE|WEEKDAY_APPR_PROCESS_START|HOUR_APPR_PROCESS_START|FLAG_LAST_APPL_PER_CONTRACT|NFLAG_LAST_APPL_IN_DAY|RATE_DOWN_PAYMENT|RATE_INTEREST_PRIMARY|RATE_INTEREST_PRIVILEGED|NAME_CASH_LOAN_PURP

In [3]:
previous_application.count()

1670214

In [12]:
previous_application = spark.sql("""
                                Select
                                    SK_ID_PREV as PK_AGG_PREVIOUS,
                                    SK_ID_CURR,
                                    NAME_CONTRACT_TYPE as FC_NAME_CONTRACT_TYPE_PREVIOUS,
                                    AMT_ANNUITY as FVL_AMT_ANNUITY_PREVIOUS,
                                    AMT_APPLICATION as FVL_AMT_APPLICATION_PREVIOUS,
                                    AMT_CREDIT as FVL_AMT_CREDIT_PREVIOUS,
                                    AMT_DOWN_PAYMENT as FVL_AMT_DOWN_PAYMENT_PREVIOUS,
                                    AMT_GOODS_PRICE as FVL_AMT_GOODS_PRICE_PREVIOUS,
                                    WEEKDAY_APPR_PROCESS_START as FC_WEEKDAY_APPR_PROCESS_START_PREVIOUS,
                                    HOUR_APPR_PROCESS_START as FVL_HOUR_APPR_PROCESS_START_PREVIOUS,
                                    NFLAG_LAST_APPL_IN_DAY as FVL_NFLAG_LAST_APPL_IN_DAY_PREVIOUS,
                                    RATE_DOWN_PAYMENT as FVL_RATE_DOWN_PAYMENT_PREVIOUS,
                                    RATE_INTEREST_PRIMARY as FVL_RATE_INTEREST_PRIMARY_PREVIOUS,
                                    RATE_INTEREST_PRIVILEGED as FVL_RATE_INTEREST_PRIVILEGED_PREVIOUS,
                                    NAME_CASH_LOAN_PURPOSE as FC_NAME_CASH_LOAN_PURPOSE_PREVIOUS,
                                    NAME_CONTRACT_STATUS as FC_NAME_CONTRACT_STATUS_PREVIOUS,
                                    NAME_PAYMENT_TYPE as FC_NAME_PAYMENT_TYPE_PREVIOUS,
                                    CODE_REJECT_REASON as FC_CODE_REJECT_REASON_PREVIOUS,
                                    NAME_TYPE_SUITE as FC_NAME_TYPE_SUITE_PREVIOUS,
                                    NAME_CLIENT_TYPE as FC_NAME_CLIENT_TYPE_PREVIOUS,
                                    NAME_GOODS_CATEGORY as FC_NAME_GOODS_CATEGORY_PREVIOUS,
                                    NAME_PORTFOLIO as FC_NAME_PORTFOLIO_PREVIOUS,
                                    CHANNEL_TYPE as FC_CHANNEL_TYPE_PREVIOUS,
                                    SELLERPLACE_AREA as FVL_SELLERPLACE_AREA_PREVIOUS,
                                    CNT_PAYMENT as FVL_CNT_PAYMENT_PREVIOUS,
                                    NAME_SELLER_INDUSTRY as FC_NAME_SELLER_INDUSTRY_PREVIOUS,
                                    NAME_YIELD_GROUP as FC_NAME_YIELD_GROUP_PREVIOUS,
                                    PRODUCT_COMBINATION as FC_PRODUCT_COMBINATION_PREVIOUS,
                                    DAYS_FIRST_DRAWING as FVL_DAYS_FIRST_DRAWING_PREVIOUS,
                                    DAYS_FIRST_DUE as FVL_DAYS_FIRST_DUE_PREVIOUS,
                                    DAYS_LAST_DUE_1ST_VERSION as FVL_DAYS_LAST_DUE_1ST_VERSION_PREVIOUS,
                                    DAYS_LAST_DUE as FVL_DAYS_LAST_DUE_PREVIOUS,
                                    DAYS_TERMINATION as FVL_DAYS_TERMINATION_PREVIOUS,
                                    NFLAG_INSURED_ON_APPROVAL as FVL_NFLAG_INSURED_ON_APPROVAL_PREVIOUS,
                                    cast(date_add('2023-12-01', cast(DAYS_DECISION as int)) as timestamp) as PK_DATREF_PREVIOUS,
                                    substr(translate(cast(date_add('2023-12-01', cast(DAYS_DECISION as int)) as string),'-',''),1,6) as PK_DATPART_PREVIOUS,
                                    DAYS_DECISION                                    
                                from 
                                    previous_application
                            """)

# Retirando valores nulos
previous_application = previous_application.where(col("DAYS_DECISION").isNotNull())

# Filtrando somente histórico necessário (15 meses)
stage01 = previous_application.where(col("DAYS_DECISION") >= -2922)
stage01 = stage01.drop("DAYS_DECISION")

stage01.createOrReplaceTempView("stage01")
stage01.count()

1670214

In [13]:
stage01.show(5, truncate=False)

+---------------+----------+------------------------------+------------------------+----------------------------+-----------------------+-----------------------------+----------------------------+--------------------------------------+------------------------------------+-----------------------------------+------------------------------+----------------------------------+-------------------------------------+----------------------------------+--------------------------------+-----------------------------+------------------------------+---------------------------+----------------------------+-------------------------------+--------------------------+------------------------+-----------------------------+------------------------+--------------------------------+----------------------------+-------------------------------+-------------------------------+---------------------------+--------------------------------------+--------------------------+-----------------------------+-------------

In [14]:
spark.sql("""
            select
                PK_DATPART_PREVIOUS,
                count(*) as Volume
            from stage01
            group by 1
            order by  1 desc
""").show(200)

[Stage 23:>                                                         (0 + 6) / 6]

+-------------------+------+
|PK_DATPART_PREVIOUS|Volume|
+-------------------+------+
|             202311| 37267|
|             202310| 28174|
|             202309| 24075|
|             202308| 35967|
|             202307| 44656|
|             202306| 53404|
|             202305| 61427|
|             202304| 60449|
|             202303| 63114|
|             202302| 53837|
|             202301| 56505|
|             202212| 54865|
|             202211| 46795|
|             202210| 42936|
|             202209| 38386|
|             202208| 36687|
|             202207| 35126|
|             202206| 31100|
|             202205| 28769|
|             202204| 26089|
|             202203| 26177|
|             202202| 22559|
|             202201| 24843|
|             202112| 25631|
|             202111| 22711|
|             202110| 22430|
|             202109| 20171|
|             202108| 19838|
|             202107| 18845|
|             202106| 17598|
|             202105| 17015|
|             

In [15]:
spark.sql("""
            select
                PK_DATPART_PREVIOUS,
                count(*) as Volume
            from stage01
            group by 1
            order by  1 desc
""").count()

96

#### Salvando tabela particionada (Parquet)

In [16]:
nm_path = '/data/processed/previous_app/'
stage01.write.partitionBy('PK_DATPART_PREVIOUS').parquet(nm_path, mode='overwrite')

In [17]:
spark.stop()